# 06. Supplier Risk, Dynamic Safety Stock & Dual Sourcing — ROLLINGMRP

This notebook demonstrates **Phase 5** capabilities:
1. **Supplier Reliability Scorecards & Performance Tiers** (OTD, Fill Rate, Quality, Lead Time Stability)
2. **Static vs. Risk-Adjusted Dynamic Safety Stock** ($SS = Z \times \sigma_D \times \sqrt{L+R} \times \text{risk\_multiplier}$)
3. **Proactive Lead Time Buffering** ($L_{\text{effective}} = L_{\text{quoted}} + \lceil k \times \sigma_L \rceil$)
4. **Dual-Sourcing Order Allocation & Resiliency Analysis** (Primary vs. Backup trade-off)
5. **Supplier Disruption Sensitivity Simulation**

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
current_dir = Path.cwd().resolve()
repo_root = current_dir if (current_dir / "src").exists() else current_dir.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.integration.data_generator import generate_synthetic_data
from src.risk.supplier import SupplierReliabilityScorer, DualSourcingManager, SupplierTier
from src.risk.safety_stock import DynamicSafetyStockEngine

print(f"Supplier Risk Environment initialized. Root: {repo_root}")

## 1. Supplier Portfolio Scorecards & Performance Tiers
Evaluating the 4 suppliers from our synthetic factory dataset across On-Time Delivery, Fill Rate, Quality Rejection, and Delivery Variability.

In [ ]:
raw_data = generate_synthetic_data(seed=42)
suppliers_df = raw_data["suppliers"]

scorer = SupplierReliabilityScorer()
scorecards = scorer.evaluate_supplier_portfolio(suppliers_df)
scorecard_df = pd.DataFrame([s.to_dict() for s in scorecards])

print("Supplier Performance Scorecard:")
cols = ["supplier_id", "name", "composite_score", "tier", "on_time_delivery_pct", "fill_rate_pct", "lead_time_stability_pct"]
print(scorecard_df[cols].to_string(index=False))

## 2. Supplier Composite Score Visual Breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2ca02c" if s >= 75 else ("#ff7f0e" if s >= 60 else "#d62728") for s in scorecard_df["composite_score"]]

bars = ax.barh(scorecard_df["name"], scorecard_df["composite_score"], color=colors, alpha=0.85)
ax.axvline(90, color="darkgreen", ls=":", label="Preferred (>=90)")
ax.axvline(75, color="green", ls="--", label="Approved (>=75)")
ax.axvline(60, color="orange", ls="--", label="Conditional (>=60)")

for bar in bars:
    w = bar.get_width()
    ax.text(w + 1, bar.get_y() + bar.get_height()/2, f"{w:.1f}", va="center", fontweight="bold")

ax.set_xlim(0, 105)
ax.set_title("Supplier Composite Reliability Scores & Risk Tiers", fontsize=13, fontweight="bold")
ax.set_xlabel("Composite Score (0 - 100)", fontsize=11)
ax.grid(True, alpha=0.3, axis="x")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. Static vs. Dynamic Risk-Adjusted Safety Stock
Comparing baseline safety stocks vs. dynamic buffers adjusted for vendor delivery risk.

In [ ]:
scorecard_map = {s.supplier_id: s for s in scorecards}
ss_engine = DynamicSafetyStockEngine()
items_df = raw_data["items"][raw_data["items"]["type"] == "raw_material"]

recs = ss_engine.evaluate_item_portfolio(items_df, scorecard_map)
recs_df = pd.DataFrame([r.to_dict() for r in recs])

print("Sample Risk-Adjusted Safety Stock Recommendations (Raw Materials):")
print(recs_df[["item_id", "supplier_id", "supplier_score", "base_safety_stock", "risk_multiplier", "dynamic_safety_stock", "lead_time_buffer_days"]].head(10).to_string(index=False))

## 4. Dual-Sourcing Allocation & Risk Reduction
Evaluating dual-sourcing splits between primary vendor (e.g. SUP-01) and backup vendor (e.g. SUP-02) across varying order volumes.

In [ ]:
dual_mgr = DualSourcingManager(min_split_pct=20.0, max_primary_pct=80.0)
sup_primary = scorecard_map["SUP-01"]
sup_backup = scorecard_map["SUP-02"]

split = dual_mgr.calculate_order_split(
    item_id="RM-01",
    total_quantity=1000.0,
    primary_metrics=sup_primary,
    backup_metrics=sup_backup,
    primary_unit_cost=15.0,
    backup_unit_cost=17.5,
)

print("=== Dual-Sourcing Allocation Recommendation ===")
print(f"Item: {split.item_id}")
print(f"Total Required Quantity: {split.total_order_quantity:.0f} units")
print(f"Primary ({split.primary_supplier_name}): {split.primary_share_pct:.1f}% ({split.primary_quantity:.0f} units)")
print(f"Backup  ({split.backup_supplier_name}): {split.backup_share_pct:.1f}% ({split.backup_quantity:.0f} units)")
print(f"Single-Source Cost: ${split.single_source_cost:,.2f}")
print(f"Dual-Source Cost:   ${split.dual_source_cost:,.2f} (Premium: +{split.cost_premium_pct:.1f}%)")
print(f"Supply Chain Delivery Risk Reduction: {split.portfolio_risk_reduction_pct:.1f}%")